# Notebook 3 — Baseline Evaluation and Comparative Analysis

 Purpose:
 - load baseline outputs from Notebook 2
 - compute deterministic similarity metrics
 - compute structural completeness as an aggregated score
 - compute an LLM-as-a-judge score
 - normalize selected metrics for comparison
 - generate a Plotly radar chart
 - log final metrics and artifacts to MLflow

In [ ]:
from __future__ import annotations

import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import openai
from groq import Groq

from dotenv import load_dotenv
from openai import OpenAI

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score

import plotly.graph_objects as go

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENV_PATH = PROJECT_ROOT / ".env"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = OUTPUTS_DIR / "reports"
FIGURES_DIR = OUTPUTS_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "phi4_vs_gpt35_baseline")

if not OPENAI_API_KEY:
    raise EnvironmentError("Missing OPENAI_API_KEY")

judge_client = OpenAI(api_key=OPENAI_API_KEY)

print("Environment ready.")
print("MLflow version:", mlflow.__version__)
print("OpenAI version:", openai.__version__)

Environment ready.
MLflow version: 3.10.1
OpenAI version: 2.29.0


In [7]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError("MLflow experiment could not be resolved.")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name)

Tracking URI: http://127.0.0.1:5000
Experiment: phi4_vs_gpt35_baseline


In [8]:
CASE_ID = "BIAT_2021_2024_v1"
OUTPUTS_CSV_PATH = OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}.csv"

if not OUTPUTS_CSV_PATH.exists():
    raise FileNotFoundError(f"Missing outputs file: {OUTPUTS_CSV_PATH}")

outputs_df = pd.read_csv(OUTPUTS_CSV_PATH)

required_columns = [
    "case_id",
    "model_label",
    "provider",
    "latency_sec",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "output_text",
    "reference_output",
]

missing_cols = [c for c in required_columns if c not in outputs_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

outputs_df["latency_ms"] = outputs_df["latency_sec"] * 1000

display(outputs_df[["model_label", "provider", "latency_sec", "total_tokens"]])

,model_label,provider,latency_sec,total_tokens
0,Phi-4,azure_openai_compatible,32.2288,1208
1,GPT-3.5,openai,12.1940,914


## Metric helpers

In [9]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1


def compute_rouge_l(reference: str, prediction: str) -> float:
    return rouge.score(reference, prediction)["rougeL"].fmeasure


def compute_bleu(reference: str, prediction: str) -> float:
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()

    if not reference_tokens or not prediction_tokens:
        return 0.0

    return sentence_bleu([reference_tokens], prediction_tokens, smoothing_function=smooth)


def compute_output_length(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0


def normalized_length_score(prediction: str, reference: str) -> float:
    pred_len = compute_output_length(prediction)
    ref_len = compute_output_length(reference)

    if ref_len == 0:
        return 0.0

    score = 1 - abs(pred_len - ref_len) / ref_len
    return float(max(0.0, min(1.0, score)))

## Structural checks and aggregated structure score

In [10]:
def contains_profitability(text: str) -> int:
    patterns = [r"\bprofitability\b", r"\bnet income\b", r"\bearnings\b"]
    return int(any(re.search(p, text, flags=re.IGNORECASE) for p in patterns))


def contains_conclusion(text: str) -> int:
    patterns = [r"\bconclusion\b", r"\boverall\b", r"\bin summary\b"]
    return int(any(re.search(p, text, flags=re.IGNORECASE) for p in patterns))


def contains_risk_discussion(text: str) -> int:
    patterns = [r"\brisk\b", r"\bcost of risk\b", r"\bcredit risk\b"]
    return int(any(re.search(p, text, flags=re.IGNORECASE) for p in patterns))


def structure_components(text: str) -> dict:
    has_profitability = contains_profitability(text)
    has_conclusion = contains_conclusion(text)
    has_risk_discussion = contains_risk_discussion(text)

    total = has_profitability + has_conclusion + has_risk_discussion
    score = total / 3

    return {
        "has_profitability": has_profitability,
        "has_conclusion": has_conclusion,
        "has_risk_discussion": has_risk_discussion,
        "structure_points": total,
        "structure_coverage": f"{total}/3",
        "structure_score": score,
    }

## Compute deterministic metrics

In [11]:
outputs_df["rougeL"] = outputs_df.apply(
    lambda row: compute_rouge_l(row["reference_output"], row["output_text"]),
    axis=1,
)

outputs_df["bleu"] = outputs_df.apply(
    lambda row: compute_bleu(row["reference_output"], row["output_text"]),
    axis=1,
)

outputs_df["length_score"] = outputs_df.apply(
    lambda row: normalized_length_score(row["output_text"], row["reference_output"]),
    axis=1,
)

structure_df = outputs_df["output_text"].apply(structure_components).apply(pd.Series)
outputs_df = pd.concat([outputs_df, structure_df], axis=1)

display(
    outputs_df[
        [
            "model_label",
            "rougeL",
            "bleu",
            "length_score",
            "structure_coverage",
            "structure_score",
        ]
    ]
)

,model_label,rougeL,bleu,length_score,structure_coverage,structure_score
0,Phi-4,0.332130,0.143640,0.387302,3/3,1.0
1,GPT-3.5,0.349014,0.144195,0.933333,3/3,1.0


## Compute BERTScore 

In [12]:
P, R, F1 = bertscore_score(
    outputs_df["output_text"].tolist(),
    outputs_df["reference_output"].tolist(),
    lang="en",
    verbose=False,
)

outputs_df["bert_f1"] = [float(x) for x in F1]

display(outputs_df[["model_label", "rougeL", "bleu", "bert_f1"]])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,model_label,rougeL,bleu,bert_f1
0,Phi-4,0.332130,0.143640,0.870665
1,GPT-3.5,0.349014,0.144195,0.899563


## LLM-as-a-judge

In [13]:
def judge_output_with_llm(
    reference: str,
    prediction: str,
    client: OpenAI,
    judge_model: str = "gpt-4.1-mini",
):
    prompt = f"""
You are an expert evaluator for financial analysis reports.

Your task is to compare a model-generated answer to a reference answer.

Score the generated answer on a scale from 0 to 1 according to:
1. Correctness of financial interpretation
2. Coverage of important points
3. Clarity and structure
4. Faithfulness to the provided reference

Return ONLY a valid JSON object in this exact format:
{{
  "score": 0.0,
  "reason": "short explanation"
}}

Reference answer:
\"\"\"
{reference}
\"\"\"

Generated answer:
\"\"\"
{prediction}
\"\"\"
""".strip()

    response = client.chat.completions.create(
        model=judge_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=250,
    )

    raw_text = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw_text)
        score = float(parsed["score"])
        reason = parsed["reason"]
    except Exception:
        score = np.nan
        reason = raw_text

    return score, reason

In [14]:
judge_results = outputs_df.apply(
    lambda row: judge_output_with_llm(
        reference=row["reference_output"],
        prediction=row["output_text"],
        client=judge_client,
        judge_model="gpt-4.1-mini",
    ),
    axis=1,
)

outputs_df[["judge_score", "judge_reason"]] = pd.DataFrame(
    judge_results.tolist(),
    index=outputs_df.index,
)

display(outputs_df[["model_label", "judge_score", "judge_reason"]])

,model_label,judge_score,judge_reason
0,Phi-4,1.0,The generated answer accurately interprets the...
1,GPT-3.5,0.9,The generated answer correctly interprets the ...


## Normalize latency for radar

In [15]:
def invert_minmax(series: pd.Series) -> pd.Series:
    series = series.astype(float)
    min_v = series.min()
    max_v = series.max()

    if max_v == min_v:
        return pd.Series([1.0] * len(series), index=series.index)

    normalized = (series - min_v) / (max_v - min_v)
    inverted = 1 - normalized
    return inverted.clip(0, 1)

In [16]:
comparison_df = outputs_df[
    [
        "case_id",
        "model_label",
        "provider",
        "latency_ms",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
        "rougeL",
        "bleu",
        "bert_f1",
        "length_score",
        "has_profitability",
        "has_conclusion",
        "has_risk_discussion",
        "structure_points",
        "structure_coverage",
        "structure_score",
        "judge_score",
        "judge_reason",
    ]
].copy()

display(comparison_df)

,case_id,model_label,provider,latency_ms,prompt_tokens,completion_tokens,total_tokens,rougeL,bleu,bert_f1,length_score,has_profitability,has_conclusion,has_risk_discussion,structure_points,structure_coverage,structure_score,judge_score,judge_reason
0,BIAT_2021_2024_v1,Phi-4,azure_openai_compatible,32228.8,433,775,1208,0.332130,0.143640,0.870665,0.387302,1,1,1,3,3/3,1.0,1.0,The generated answer accurately interprets the...
1,BIAT_2021_2024_v1,GPT-3.5,openai,12194.0,434,480,914,0.349014,0.144195,0.899563,0.933333,1,1,1,3,3/3,1.0,0.9,The generated answer correctly interprets the ...


## Build radar dataframe

In [17]:
radar_df = comparison_df.copy()
radar_df["latency_score"] = invert_minmax(radar_df["latency_ms"])

radar_metrics = [
    "rougeL",
    "bleu",
    "bert_f1",
    "length_score",
    "structure_score",
    "judge_score",
    "latency_score",
]

for col in radar_metrics:
    radar_df[col] = pd.to_numeric(radar_df[col], errors="coerce").fillna(0.0).clip(0, 1)

display(radar_df[["model_label"] + radar_metrics])

,model_label,rougeL,bleu,bert_f1,length_score,structure_score,judge_score,latency_score
0,Phi-4,0.332130,0.143640,0.870665,0.387302,1.0,1.0,0.0
1,GPT-3.5,0.349014,0.144195,0.899563,0.933333,1.0,0.9,1.0


In [18]:
def plot_plotly_radar(df: pd.DataFrame, model_col: str, metrics: list[str], title: str):
    categories = metrics + [metrics[0]]

    fig = go.Figure()

    for _, row in df.iterrows():
        values = row[metrics].tolist()
        values += values[:1]

        fig.add_trace(
            go.Scatterpolar(
                r=values,
                theta=categories,
                fill="toself",
                name=row[model_col],
            )
        )

    fig.update_layout(
        title=title,
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1],
            )
        ),
        showlegend=True,
    )

    return fig

In [26]:
radar_fig = plot_plotly_radar(
    radar_df,
    model_col="model_label",
    metrics=radar_metrics,
    title="Baseline Evaluation Radar Chart",
)

# Show interactive chart inside the notebook
radar_fig.show()

# Save chart
radar_html_path = FIGURES_DIR / f"radar_comparison_{CASE_ID}.html"
radar_png_path = FIGURES_DIR / f"radar_comparison_{CASE_ID}.png"

radar_fig.write_html(str(radar_html_path))
radar_fig.write_image(str(radar_png_path))

print("Saved:", radar_html_path)
print("Saved:", radar_png_path)

Saved: C:\Users\BRHN\Desktop\SLM-evals\outputs\figures\radar_comparison_BIAT_2021_2024_v1.html
Saved: C:\Users\BRHN\Desktop\SLM-evals\outputs\figures\radar_comparison_BIAT_2021_2024_v1.png


## MLflow quality scoring

In [21]:
from mlflow.genai.scorers import Correctness
from mlflow.genai import scorer

@scorer
def structural_completeness(*, outputs=None, **kwargs):
    return structure_components(outputs)["structure_score"]

eval_data = outputs_df.apply(
    lambda row: {
        "inputs": {"case_id": row["case_id"], "model_label": row["model_label"]},
        "outputs": row["output_text"],
        "expectations": {"expected_response": row["reference_output"]},
    },
    axis=1,
).tolist()

In [22]:
with mlflow.start_run(run_name=f"nb3_evaluation_{CASE_ID}") as run:
    eval_run_id = run.info.run_id

    mlflow.log_params({
        "phase": "evaluation",
        "case_id": CASE_ID,
        "rows_evaluated": len(outputs_df),
        "judge_model_local": "gpt-4.1-mini",
        "source_outputs_csv": str(OUTPUTS_CSV_PATH),
    })

    mlflow.set_tags({
        "notebook": "nb3",
        "stage": "evaluation",
        "comparison": "phi4_vs_gpt35",
        "task": "financial_analysis",
    })

    eval_result = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[
            Correctness(model="openai:/gpt-4o-mini"),
            structural_completeness,
        ],
    )

2026/04/06 00:33:38 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?] 

## Persist reports

In [23]:
comparison_csv_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}.csv"
comparison_json_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}.json"

comparison_df.to_csv(comparison_csv_path, index=False, encoding="utf-8")
comparison_df.to_json(comparison_json_path, orient="records", force_ascii=False, indent=2)

print("Saved:", comparison_csv_path)
print("Saved:", comparison_json_path)

Saved: C:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_BIAT_2021_2024_v1.csv
Saved: C:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_BIAT_2021_2024_v1.json


## Log final artifacts to MLflow

In [24]:
with mlflow.start_run(run_id=eval_run_id):
    mlflow.log_artifact(str(comparison_csv_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(comparison_json_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(radar_html_path), artifact_path="evaluation_figures")
    mlflow.log_artifact(str(radar_png_path), artifact_path="evaluation_figures")

    for _, row in comparison_df.iterrows():
        prefix = row["model_label"].lower().replace(".", "").replace("-", "_")

        mlflow.log_metric(f"{prefix}_rougeL", float(row["rougeL"]))
        mlflow.log_metric(f"{prefix}_bleu", float(row["bleu"]))
        mlflow.log_metric(f"{prefix}_bert_f1", float(row["bert_f1"]))
        mlflow.log_metric(f"{prefix}_length_score", float(row["length_score"]))
        mlflow.log_metric(f"{prefix}_structure_score", float(row["structure_score"]))
        mlflow.log_metric(f"{prefix}_judge_score", float(row["judge_score"]))
        mlflow.log_metric(f"{prefix}_latency_ms", float(row["latency_ms"]))

In [15]:
import re
import mlflow
from mlflow.genai import scorer

def extract_numbers(text: str):
    return re.findall(r"\(?\d[\d,]*\.?\d*%?\)?", text)

@scorer
def word_count(*, outputs):
    return len(outputs.split())

@scorer
def section_coverage(*, outputs, expectations):
    text = outputs.lower()
    required_sections = expectations["required_sections"]
    hits = sum(1 for sec in required_sections if sec in text)
    return hits / len(required_sections)

@scorer
def has_conclusion(*, outputs):
    return "conclusion" in outputs.lower()

@scorer
def unsupported_number_rate(*, inputs, outputs):
    source_numbers = set(extract_numbers(inputs["financial_data"]))
    output_numbers = extract_numbers(outputs)

    if not output_numbers:
        return 0.0

    unsupported = sum(1 for n in output_numbers if n not in source_numbers)
    return unsupported / len(output_numbers)

@scorer
def reference_number_alignment(*, outputs, expectations):
    ref_numbers = set(extract_numbers(expectations["reference_report"]))
    output_numbers = set(extract_numbers(outputs))

    if not ref_numbers:
        return 0.0

    return len(ref_numbers & output_numbers) / len(ref_numbers)

## the MLflow evaluation

In [16]:
mlflow.set_experiment("SLM Financial Report Evaluation")

mlflow_eval_result = mlflow.genai.evaluate(
    data=eval_rows,
    scorers=[
        word_count,
        section_coverage,
        has_conclusion,
        unsupported_number_rate,
        reference_number_alignment,
    ],
)

2026/03/26 22:26:37 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?] 


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: caring-roo-625
  Run ID: 903927e372ff454a96531550e4f197c5

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



In [17]:
# 1. Aggregate metrics
mlflow_eval_result.metrics

{'reference_number_alignment/mean': np.float64(0.8400000000000001),
 'unsupported_number_rate/mean': np.float64(0.3776422764227642),
 'section_coverage/mean': np.float64(1.0),
 'has_conclusion/mean': np.float64(1.0),
 'word_count/mean': np.float64(604.0)}

In [18]:
# 2. Detailed row-level results
mlflow_eval_result.result_df

,trace_id,reference_number_alignment/value,unsupported_number_rate/value,section_coverage/value,has_conclusion/value,word_count/value,reference_report/value,required_sections/value,model_name/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-6dab164c06435307b4c242b89400df56,0.88,0.300000,1.0,True,477.0,\n\nFinancial Analysis – Banque Internationale...,None,Phi-4,"{""info"": {""trace_id"": ""tr-6dab164c06435307b4c2...",None,OK,1774560398059,0,{'user_input': ' Analyze the following bank fi...,# Financial Analysis Report\n\n## Executive Su...,"{'mlflow.user': 'BRHN', 'mlflow.source.name': ...",{'mlflow.artifactLocation': 'file:///C:/Users/...,"[{'trace_id': 'basWTAZDUwe0wkK4lADfVg==', 'spa...",[{'assessment_id': 'a-e1dc966aec5a427e87bf7d9e...
1,tr-1ced87cda7d0164ee3994ea1d9379df6,0.80,0.455285,1.0,True,731.0,\n\nFinancial Analysis – Banque Internationale...,None,ChatGPT-web-5.0,"{""info"": {""trace_id"": ""tr-1ced87cda7d0164ee399...",None,OK,1774560398059,2,{'user_input': ' Analyze the following bank fi...,\n# Executive summary\n\nBetween 2021 and 2024...,"{'mlflow.user': 'BRHN', 'mlflow.source.name': ...",{'mlflow.artifactLocation': 'file:///C:/Users/...,"[{'trace_id': 'HO2HzafQFk7jmU6h2Ted9g==', 'spa...",[{'assessment_id': 'a-7ba6e1aa7a8c49d5935c5482...


In [19]:
manual_scores = pd.DataFrame([
    {
        "model": "Phi-4",
        "numerical_fidelity": None,
        "coverage": None,
        "analytical_depth": None,
        "groundedness": None,
        "writing_quality": None,
        "notes": ""
    },
    {
        "model": "GPT-manual",
        "numerical_fidelity": None,
        "coverage": None,
        "analytical_depth": None,
        "groundedness": None,
        "writing_quality": None,
        "notes": ""
    }
])

manual_scores

,model,numerical_fidelity,coverage,analytical_depth,groundedness,writing_quality,notes
0,Phi-4,None,None,None,None,None,
1,GPT-manual,None,None,None,None,None,
